# 05 Column Role Leakage Timing Audit

Conservative, column-by-column audit for the 100원딜 OTT churn project. This notebook does not perform modeling, prediction, SHAP, Optuna, feature engineering, row exclusion, duplicate removal, or model-ready dataset creation.

In [1]:
from pathlib import Path
from datetime import datetime
from zipfile import ZipFile, ZIP_DEFLATED
from difflib import SequenceMatcher
import json
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

STEP = "05_column_role_leakage_timing_audit_260513"
NOW = datetime.now()
RUN_TS = NOW.strftime("%Y%m%d_%H%M%S")
CWD = Path.cwd().resolve()
SOURCE_NAME = "(광일)Membership_v2_with_derived_features.csv"
PARK = None
for cand in [CWD, *CWD.parents]:
    if (cand / "data" / SOURCE_NAME).exists():
        PARK = cand.resolve()
        break
    if (cand / "park.ingyeom" / "data" / SOURCE_NAME).exists():
        PARK = (cand / "park.ingyeom").resolve()
        break
if PARK is None:
    PARK = (CWD / "park.ingyeom").resolve()
ROOT = PARK.parent.resolve()
SRC = PARK / "data" / "(광일)Membership_v2_with_derived_features.csv"
NB_PATH = PARK / "notebook" / STEP / f"{STEP}.ipynb"
OUT_BASE = PARK / "reports" / "audits" / STEP
NOTE = PARK / "note.md"
ZIP_DIR = PARK / "zip"
ZIP_PATH = ZIP_DIR / f"{STEP}_review_package.zip"
PREV = {
    "01": PARK / "reports" / "audits" / "01_data_contract_260513",
    "02": PARK / "reports" / "audits" / "02_target_score_orientation_260513",
    "03": PARK / "reports" / "audits" / "03_observation_window_policy_260513",
    "04": PARK / "reports" / "audits" / "04_promotion_split_260513",
}
PREV_FILES = [
    "01_column_inventory.csv", "01_data_contract_summary.csv", "01_user_key_duplicate_audit.csv", "01_final_checks.csv",
    "02_analysis_unit_contract.csv", "02_target_contract.csv", "02_score_orientation_policy.csv", "02_score_naming_convention.csv", "02_open_risks_for_next_steps.csv", "02_final_checks.csv",
    "03_week_feature_inventory.csv", "03_timing_allowed_for_modeling_policy.csv", "03_open_risks_for_next_steps.csv", "03_final_checks.csv",
    "04_promotion_split_contract.csv", "04_groupwise_modeling_policy.csv", "04_open_risks_for_next_steps.csv", "04_final_checks.csv",
]
REQ = [
    "05_input_consistency_check.csv", "05_full_column_inventory.csv", "05_column_role_dictionary.csv",
    "05_timing_audit.csv", "05_leakage_suspect_audit.csv", "05_human_review_required_columns.csv",
    "05_baseline_ladder_feature_family_policy.csv", "05_recommended_feature_set_contracts.csv",
    "05_forbidden_drop_columns.csv", "05_review_required_columns.csv", "05_conservative_safe_candidate_columns.csv",
    "05_redundancy_and_naming_risk_audit.csv", "05_safe_unsafe_wording.csv", "05_open_risks_for_next_steps.csv",
]

def inside(child, parent):
    try:
        Path(child).resolve().relative_to(Path(parent).resolve())
        return True
    except ValueError:
        return False

for p in [NB_PATH.parent, OUT_BASE, ZIP_DIR]:
    p.mkdir(parents=True, exist_ok=True)
payload = [p for p in OUT_BASE.iterdir() if p.name != ".ipynb_checkpoints"] if OUT_BASE.exists() else []
if payload:
    OUT = OUT_BASE / f"run_{RUN_TS}"
    OUT.mkdir(parents=True, exist_ok=False)
    out_mode = "run_subfolder_created_because_base_output_folder_was_not_empty"
else:
    OUT = OUT_BASE
    OUT.mkdir(parents=True, exist_ok=True)
    out_mode = "base_output_folder_used"

assert inside(SRC, PARK)
assert inside(OUT, PARK)
assert inside(NB_PATH, PARK)
assert inside(ZIP_PATH, PARK)
assert SRC.exists(), f"missing source: {SRC}"
src_mtime_before = SRC.stat().st_mtime

def read_csv(path):
    last = None
    for enc in ["utf-8-sig", "utf-8", "cp949", "euc-kr"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last = e
    raise last

df = read_csv(SRC)
warnings_log = []
prev_loaded = {}
for step, folder in PREV.items():
    if not folder.exists():
        warnings_log.append({"warning_type": "missing_previous_folder", "item": str(folder.relative_to(PARK)), "detail": "previous folder missing"})
        continue
    for name in PREV_FILES:
        if not name.startswith(step):
            continue
        p = folder / name
        if p.exists():
            try:
                prev_loaded[name] = read_csv(p)
            except Exception as e:
                warnings_log.append({"warning_type": "previous_file_read_failed", "item": str(p.relative_to(PARK)), "detail": repr(e)})
        else:
            warnings_log.append({"warning_type": "missing_previous_output", "item": str(p.relative_to(PARK)), "detail": "expected previous output missing"})

def j(x):
    if isinstance(x, (dict, list)):
        return json.dumps(x, ensure_ascii=False)
    return "" if x is None else str(x)

def counts(s):
    return json.dumps({str(k): int(v) for k, v in s.value_counts(dropna=False).sort_index().items()}, ensure_ascii=False)

row_count, col_count = df.shape
reg = pd.to_datetime(df["reg_date"], errors="coerce") if "reg_date" in df else pd.Series(dtype="datetime64[ns]")
end = pd.to_datetime(df["end_date"], errors="coerce") if "end_date" in df else pd.Series(dtype="datetime64[ns]")
dur = (end - reg).dt.days if len(reg) and len(end) else pd.Series(dtype="float64")
promo_target = pd.crosstab(df["is_promotion"], df["is_repurchase"], dropna=False).to_json(force_ascii=False) if {"is_promotion", "is_repurchase"}.issubset(df.columns) else ""
cross_overlap = int((df.groupby("USER_KEY")["is_promotion"].nunique(dropna=True) > 1).sum()) if {"USER_KEY", "is_promotion"}.issubset(df.columns) else None
metrics = {
    "row_count": row_count,
    "column_count": col_count,
    "total_missing_count": int(df.isna().sum().sum()),
    "unique_USER_KEY_count": int(df["USER_KEY"].nunique()) if "USER_KEY" in df else None,
    "duplicated_USER_KEY_extra_rows": int(df.duplicated("USER_KEY").sum()) if "USER_KEY" in df else None,
    "duplicated_full_row_count": int(df.duplicated().sum()),
    "is_repurchase_value_counts": counts(df["is_repurchase"]) if "is_repurchase" in df else "",
    "is_promotion_value_counts": counts(df["is_promotion"]) if "is_promotion" in df else "",
    "promotion_x_is_repurchase_2x2": promo_target,
    "cross_promotion_USER_KEY_overlap_count": cross_overlap,
    "reg_date_parse_success": int(reg.notna().sum()) if len(reg) else None,
    "reg_date_parse_failure": int(reg.isna().sum()) if len(reg) else None,
    "end_date_parse_success": int(end.notna().sum()) if len(end) else None,
    "end_date_parse_failure": int(end.isna().sum()) if len(end) else None,
    "duration_lt_21_count": int((dur < 21).sum()) if len(dur) else None,
    "duration_eq_0_count": int((dur == 0).sum()) if len(dur) else None,
    "duration_21_to_30_count": int(((dur >= 21) & (dur <= 30)).sum()) if len(dur) else None,
    "duration_gte_21_count": int((dur >= 21).sum()) if len(dur) else None,
}

def prev_ref(metric):
    toks = [t for t in re.split(r"[^a-z0-9_]+", metric.lower()) if t]
    hits = []
    for fname, pdf in prev_loaded.items():
        for c in pdf.columns:
            if any(k in c.lower() for k in ["metric", "check", "item", "name", "column", "field"]):
                pat = "|".join(re.escape(t) for t in toks)
                if pat and pdf[c].astype(str).str.lower().str.contains(pat, na=False).any():
                    hits.append(fname)
    return "; ".join(sorted(set(hits[:3])))

pd.DataFrame([{
    "metric": k,
    "current_value": j(v),
    "previous_reference": prev_ref(k),
    "compare_status": "previous_reference_available_review_manually" if prev_ref(k) else "no_previous_reference_found",
    "warning": "" if prev_ref(k) else "No matching prior metric found by generic scan; not a silent pass."
} for k, v in metrics.items()]).to_csv(OUT / "05_input_consistency_check.csv", index=False, encoding="utf-8-sig")

special = re.compile(r"[^0-9A-Za-z가-힣_]")
kr_special = re.compile(r"[가-힣]|[^0-9A-Za-z_]")
def sample(s):
    return " | ".join("NA" if pd.isna(v) else str(v) for v in s.drop_duplicates().head(5).tolist())
inv = []
for i, c in enumerate(df.columns, 1):
    s = df[c]
    num = pd.api.types.is_numeric_dtype(s)
    miss = int(s.isna().sum())
    nun = int(s.nunique(dropna=True))
    zc = int((s == 0).sum()) if num else np.nan
    inv.append({
        "column_order": i, "column_name": c, "dtype": str(s.dtype), "missing_count": miss, "missing_rate": miss / row_count,
        "nunique": nun, "sample_values": sample(s), "min": s.min() if num and s.notna().any() else np.nan,
        "max": s.max() if num and s.notna().any() else np.nan, "mean": s.mean() if num and s.notna().any() else np.nan,
        "zero_count": zc, "zero_rate": zc / row_count if num else np.nan, "constant_flag": "yes" if nun <= 1 else "no",
        "binary_flag": "yes" if nun <= 2 and set(s.dropna().unique()).issubset({0, 1, True, False}) else "no",
        "high_cardinality_flag": "yes" if nun >= max(50, int(row_count * 0.5)) else "no",
        "contains_korean_or_special_chars": "yes" if kr_special.search(c) else "no",
        "raw_name_risk": "yes" if special.search(c) or any(ch in c for ch in [" ", "(", ")", "/", "%", "-"]) else "no",
    })
inventory = pd.DataFrame(inv)
inventory.to_csv(OUT / "05_full_column_inventory.csv", index=False, encoding="utf-8-sig")
inv_map = inventory.set_index("column_name").to_dict("index")

def n(c): return c.lower().strip()
def has(t, xs): return any(x in t for x in xs)
def wk(t, w): return bool(re.search(rf"(^|[^0-9a-z])w{w}([^0-9a-z]|$)", t)) or f"week{w}" in t or f"week_{w}" in t
def genre(t):
    return ("genre" in t or "ratio" in t) and has(t, ["genre", "drama", "movie", "comedy", "romance", "action", "thriller", "sf", "fantasy", "horror", "crime", "ratio"])

semantic_review = {"product_code", "price", "billing_method", "max_screen", "age", "gender"}
manual_review = {"USER_KEY", "product_code", "price", "billing_method", "max_screen", "reg_date", "end_date", "is_repurchase", "is_promotion", "is_churn_prevented", "recency", "total_watch_count", "total_watch_time(min)", "watch_time(min)_w1", "watch_time(min)_w2", "watch_time(min)_w3", "retention_w2_ratio", "retention_w3_ratio", "diff_between_w2_w1", "diff_between_w3_w1", "diff_between_w3_w2", "avg_ott_release_year", "new_movie_in_90d_ratio", "new_movie_in_180d_ratio", "new_movie_in_365d_ratio"}

def timing(c):
    t = n(c)
    if c in ["USER_KEY", "is_repurchase", "is_promotion"]: return "id_target_split", "id/target/split control column"
    if "repurchase" in t or "churn" in t or "cancel" in t: return "post_outcome_or_target", "target/outcome-like wording"
    if has(t, ["w4", "week4", "week_4", "after_day21", "response"]): return "response_period_forbidden", "4th-week or response-period wording"
    if c == "reg_date": return "subscription_anchor_date", "registration anchor date"
    if c == "end_date" or "duration" in t: return "all_period_ambiguous", "end date or duration timing unresolved"
    if wk(t,1) and wk(t,2) and not wk(t,3): return "week1_to_week2_change", "week1-week2 change"
    if wk(t,1) and wk(t,3): return "week1_to_week3_change", "week1-week3 change"
    if wk(t,2) and wk(t,3): return "week2_to_week3_change", "week2-week3 change"
    if wk(t,1): return "week1_observation", "explicit week1"
    if wk(t,2): return "week2_observation", "explicit week2"
    if wk(t,3): return "week3_observation", "explicit week3"
    if "total" in t or "all" in t: return "all_period_ambiguous", "total/all-period window unresolved"
    if "recency" in t or genre(t) or has(t, ["ott", "release", "new_movie", "cold_start", "content"]): return "content_window_ambiguous", "content/recency construction window unresolved"
    if has(t, ["product", "price", "billing", "screen", "age", "gender", "plan"]): return "static_or_pre_subscription", "likely static/subscription context but semantic review required"
    return "unknown", "timing not clear from name"

def classify(c):
    t = n(c); tf, ev = timing(c); const = inv_map[c]["constant_flag"] == "yes"
    role, fam, sub, sem, conf, fut = "unknown_review_required", "unknown_review", "unknown", "meaning not documented in this step", "yes", "06_common_preprocessing_final_cohort_policy_260513"
    if c == "USER_KEY": role, fam, sub, sem, conf, fut = "id", "identifier", "row_or_user_key", "identifier; not a unique-user analysis unit", "no", "never_use_as_model_feature"
    elif c == "is_repurchase": role, fam, sub, sem, conf, fut = "target", "target", "repurchase_positive_class", "target; positive class means repurchase", "no", "target_contract_established"
    elif c == "is_promotion": role, fam, sub, sem, conf, fut = "split", "split_variable", "promotion_row_split", "top-level split variable; use promotion rows/events wording", "no", "overall_model_comparison_only_if_used"
    elif c == "reg_date": role, fam, sub, sem, fut = "date_or_time_anchor", "subscription_anchor", "registration_date", "subscription start anchor, not a regular feature", "confirm_allowed_date_derivations_only"
    elif c == "end_date": role, fam, sub, sem, fut = "timing_review_required", "subscription_anchor", "end_date_ambiguous", "scheduled-vs-posthoc end date unresolved", "06_final_cohort_policy_must_resolve_end_date_timing"
    elif "repurchase" in t or "churn" in t or "cancel" in t: role, fam, sub, sem, fut = "leakage_suspect", "outcome_or_target_proxy", "target_like_name", "outcome-like wording", "must_resolve_before_modeling_or_drop"
    elif has(t, ["w4", "week4", "week_4", "after_day21", "response"]): role, fam, sub, sem, fut = "leakage_suspect", "response_period_behavior", "forbidden_after_day21", "response-period behavior forbidden", "drop_before_any_modeling"
    elif const: role, fam, sub, sem, fut = "duplicate_or_constant_drop_candidate", "data_quality", "constant_column", "constant column", "redundancy_policy"
    elif "duration" in t: role, fam, sub, sem, fut = "timing_review_required", "duration_or_end_date", "duration_ambiguous", "duration timing unresolved", "06_final_cohort_policy_must_resolve_duration_timing"
    elif "total" in t and has(t, ["watch", "usage", "time", "count"]): role, fam, sub, sem, fut = "aggregate_usage_review", "usage_aggregate", "total_period_ambiguous", "total usage window unresolved", "confirm_total_usage_window_before_modeling"
    elif "recency" in t: role, fam, sub, sem, fut = "content_recency", "content_recency", "reference_date_ambiguous", "recency reference date unresolved", "confirm_recency_reference_date"
    elif genre(t): role, fam, sub, sem, fut = "genre_ratio", "content_genre", "genre_or_ratio", "genre/content ratio window unresolved", "confirm_content_window_before_modeling"
    elif has(t, ["ott", "release", "new_movie", "cold_start", "content"]): role, fam, sub, sem, fut = "content_recency", "content", "release_or_cold_start", "content-derived window unresolved", "confirm_content_window_before_modeling"
    elif wk(t,1): role, fam, sub, sem, conf, fut = "activation", "usage_observation", "week1", "explicit week1 behavior within observation window if construction matches name", "no", "baseline_ladder_activation"
    elif wk(t,2): role, fam, sub, sem, conf, fut = "retention", "usage_observation", "week2_or_change", "explicit week2 behavior/change within observation window if construction matches name", "no", "baseline_ladder_retention_w2"
    elif wk(t,3): role, fam, sub, sem, conf, fut = "retention", "usage_observation", "week3_or_change", "explicit week3 behavior/change within observation window if construction matches name", "no", "baseline_ladder_retention_w3"
    elif has(t, ["product", "price", "billing", "screen", "age", "gender", "plan"]): role, fam, sub, sem, fut = "membership_context", "membership_context", "static_or_subscription_metadata", "membership metadata requires semantic review", "confirm_metadata_semantics_before_modeling"
    elif has(t, ["join", "signup", "acquisition", "channel"]): role, fam, sub, sem, fut = "acquisition", "acquisition", "pre_subscription_or_channel", "acquisition context requires semantic review", "confirm_acquisition_semantics_before_modeling"
    if c in semantic_review:
        role, fam, sub, sem, conf, fut = "membership_context", "membership_context", "static_or_subscription_metadata", "membership metadata requires semantic/scoring-time review", "yes", "confirm_metadata_semantics_before_modeling"
    overall = group = lm = la = lw2 = lw3 = lc = seg = "review"
    no_roles = {"id", "target", "date_or_time_anchor", "timing_review_required", "leakage_suspect", "duplicate_or_constant_drop_candidate", "drop_candidate"}
    if role in no_roles:
        overall = group = lm = la = lw2 = lw3 = lc = "no"; seg = "no" if role in {"id", "target", "leakage_suspect"} else "review"
    if role == "split":
        overall, group, lm, la, lw2, lw3, lc, seg = "yes", "no", "no", "no", "no", "no", "no", "review"
    if role == "membership_context": overall = group = lm = "review"
    if role == "activation" and tf == "week1_observation": overall = group = la = seg = "yes"
    if role == "retention" and tf in ["week2_observation", "week1_to_week2_change"]: overall = group = lw2 = seg = "yes"
    if role == "retention" and tf in ["week3_observation", "week1_to_week3_change", "week2_to_week3_change"]: overall = group = lw3 = seg = "yes"
    if role in ["content", "content_recency", "genre_ratio", "aggregate_usage_review"]: overall = group = lc = seg = "review"
    reason = f"{sem}. Evidence: {ev}. Conservative rule: uncertain columns remain review, not approved."
    return dict(primary_role=role, feature_family=fam, sub_family=sub, source_semantics_guess=sem, timing_family=tf,
        allowed_for_overall_model_candidate=overall, allowed_for_groupwise_model_candidate=group,
        allowed_for_baseline_ladder_membership_only=lm, allowed_for_baseline_ladder_activation=la,
        allowed_for_baseline_ladder_retention_w2=lw2, allowed_for_baseline_ladder_retention_w3=lw3,
        allowed_for_baseline_ladder_content=lc, allowed_for_segment_design_candidate=seg, reason=reason,
        required_user_confirmation=conf, future_step_to_resolve=fut)

roles = pd.DataFrame([{"column_order": inv_map[c]["column_order"], "column_name": c, **classify(c)} for c in df.columns])
timing_rows = []
for c in df.columns:
    rr = roles.set_index("column_name").loc[c]
    allowed = "yes" if rr.timing_family in ["week1_observation", "week2_observation", "week3_observation", "week1_to_week2_change", "week1_to_week3_change", "week2_to_week3_change"] else "review"
    if rr.timing_family in ["response_period_forbidden", "post_outcome_or_target"] or c in ["USER_KEY", "is_repurchase"]: allowed = "no"
    if c == "is_promotion": allowed = "review"
    ev = timing(c)[1]
    timing_rows.append({"column_name": c, "timing_family": rr.timing_family, "allowed_by_observation_window_policy": allowed, "reason": rr.reason, "evidence_from_column_name": ev, "evidence_from_previous_step": "Step 03 contract: day 0 to day 20 equals week1 to week3; day21 onward is response period.", "needs_human_confirmation": rr.required_user_confirmation})
timing_audit = pd.DataFrame(timing_rows)
roles.to_csv(OUT / "05_column_role_dictionary.csv", index=False, encoding="utf-8-sig")
timing_audit.to_csv(OUT / "05_timing_audit.csv", index=False, encoding="utf-8-sig")

def leak(c, rr):
    t = n(c)
    if c == "USER_KEY": return "critical", "id_leakage", "Identifier can memorize users/events."
    if c == "is_repurchase" or "repurchase" in t: return "critical", "target_leakage", "Target-like column."
    if "churn" in t or "cancel" in t: return "high", "target_proxy", "Churn/cancel/prevention wording may encode outcome/intervention."
    if has(t, ["w4", "week4", "after_day21", "response"]): return "critical", "response_period_leakage", "Response-period behavior is forbidden."
    if rr.timing_family in ["all_period_ambiguous", "content_window_ambiguous", "unknown"]: return "review", "timing_ambiguous", "Construction window/reference date unresolved."
    if inv_map[c]["constant_flag"] == "yes": return "low", "duplicate_or_constant", "Constant/drop candidate."
    if rr.primary_role in ["membership_context", "raw_membership_metadata"]: return "review", "not_leakage_but_review", "Semantic/scoring-time review required."
    return "low", "not_leakage_but_review", "Included for explicit conservative review."

leak_rows, human_rows = [], []
for _, rr in roles.iterrows():
    c, t = rr.column_name, n(rr.column_name)
    must_review = c in manual_review or any(x in t for x in ["w1", "w2", "w3", "ratio", "duration", "churn", "repurchase", "cold_start"]) or genre(t)
    if must_review or rr.allowed_for_overall_model_candidate != "yes" or rr.allowed_for_groupwise_model_candidate != "yes":
        lvl, typ, why = leak(c, rr)
        leak_rows.append({"column_name": c, "leakage_risk_level": lvl, "leakage_type": typ, "why_suspicious": why + " " + rr.reason, "can_use_in_model_now": "yes" if rr.allowed_for_overall_model_candidate == rr.allowed_for_groupwise_model_candidate == "yes" else ("no" if rr.allowed_for_overall_model_candidate == rr.allowed_for_groupwise_model_candidate == "no" else "review"), "required_resolution_before_modeling": rr.future_step_to_resolve, "recommended_action": "exclude now" if rr.allowed_for_groupwise_model_candidate == "no" else ("hold for human review" if "review" in [rr.allowed_for_overall_model_candidate, rr.allowed_for_groupwise_model_candidate] else "eligible only under stated timing contract")})
    if must_review or rr.required_user_confirmation == "yes" or "review" in [rr.allowed_for_overall_model_candidate, rr.allowed_for_groupwise_model_candidate] or has(t, ["total", "recency", "duration", "release", "new_movie", "cold_start", "content"]):
        human_rows.append({"column_name": c, "why_human_review_needed": rr.reason, "what_question_to_ask_user_or_team": "이 컬럼은 정확히 어떤 원천 데이터와 어떤 기간(day 0~20, day21 이후, 전체 기간 등)으로 산출되었는가?", "possible_safe_interpretation": "scoring day 21 이전에 알려졌고 day 0~20 관측창 또는 사전/정적 정보만 사용했다면 제한적으로 후보가 될 수 있다.", "possible_leakage_interpretation": "day21 이후 반응, 사후 확정 종료, 재구매/이탈 결과, 전체 기간 사용량을 포함했다면 제외해야 한다.", "temporary_status_for_modeling": "review" if "review" in [rr.allowed_for_overall_model_candidate, rr.allowed_for_groupwise_model_candidate] else ("no" if rr.allowed_for_groupwise_model_candidate == "no" else "eligible_under_current_name_based_policy")})
leakage = pd.DataFrame(leak_rows).drop_duplicates("column_name")
human = pd.DataFrame(human_rows).drop_duplicates("column_name")
leakage.to_csv(OUT / "05_leakage_suspect_audit.csv", index=False, encoding="utf-8-sig")
human.to_csv(OUT / "05_human_review_required_columns.csv", index=False, encoding="utf-8-sig")

def semi(s): return "; ".join(s)
families = [
    ("L0_static_membership_context", "Static/subscription metadata; semantic review required.", roles.primary_role.eq("membership_context"), "Acquisition/activation context", "static_or_pre_subscription"),
    ("L1_add_promotion_overall_only", "Adds is_promotion only for overall comparison; forbidden in groupwise models.", roles.column_name.eq("is_promotion"), "Acquisition/offer split", "split variable"),
    ("L2_activation_w1", "Explicit week-1 activity columns only.", roles.allowed_for_baseline_ladder_activation.eq("yes"), "Activation", "week1 within day0~20"),
    ("L3_retention_w2", "Explicit week-2 and w1-w2 change columns only.", roles.allowed_for_baseline_ladder_retention_w2.eq("yes"), "Retention", "week2 within day0~20"),
    ("L4_retention_w3", "Explicit week-3 and changes involving week3.", roles.allowed_for_baseline_ladder_retention_w3.eq("yes"), "Retention", "week3 within day0~20"),
    ("L5_content_genre", "Content/genre/release/cold-start/recency; timing review unless documented.", roles.primary_role.isin(["content", "content_recency", "genre_ratio", "aggregate_usage_review"]), "Retention/content preference", "content window ambiguous"),
    ("L6_review_only_features", "Review columns must not enter modeling until resolved.", roles[["allowed_for_overall_model_candidate", "allowed_for_groupwise_model_candidate"]].eq("review").any(axis=1), "Review bucket", "ambiguous"),
    ("L7_forbidden_or_drop", "IDs, target-like, response-period, constants, drop candidates.", roles[["allowed_for_overall_model_candidate", "allowed_for_groupwise_model_candidate"]].eq("no").all(axis=1), "Excluded", "forbidden/drop"),
]
ladder = []
for fam, desc, mask, aarrr, basis in families:
    sub = roles[mask]
    ladder.append({"family_name": fam, "description": desc, "allowed_columns_now": semi(sub[(sub.allowed_for_overall_model_candidate == "yes") & (sub.allowed_for_groupwise_model_candidate == "yes")].column_name.tolist()), "review_columns": semi(sub[sub[["allowed_for_overall_model_candidate", "allowed_for_groupwise_model_candidate"]].eq("review").any(axis=1)].column_name.tolist()), "forbidden_columns": semi(sub[sub[["allowed_for_overall_model_candidate", "allowed_for_groupwise_model_candidate"]].eq("no").any(axis=1)].column_name.tolist()), "required_before_use": "Human confirmation required for review columns; review columns stay out.", "relation_to_AARRR": aarrr, "timing_basis": basis})
pd.DataFrame(ladder).to_csv(OUT / "05_baseline_ladder_feature_family_policy.csv", index=False, encoding="utf-8-sig")

contracts = []
sets = ["overall_model_candidate_with_promotion", "overall_model_candidate_without_promotion", "promotion_only_model_candidate", "nonpromotion_only_model_candidate", "conservative_safe_candidate", "review_required_candidate", "forbidden_drop_columns"]
for fs in sets:
    for _, rr in roles.iterrows():
        c = rr.column_name
        if fs == "overall_model_candidate_with_promotion": base = rr.allowed_for_overall_model_candidate
        elif fs == "overall_model_candidate_without_promotion": base = "no" if c == "is_promotion" else rr.allowed_for_overall_model_candidate
        elif fs in ["promotion_only_model_candidate", "nonpromotion_only_model_candidate"]: base = rr.allowed_for_groupwise_model_candidate
        elif fs == "conservative_safe_candidate": base = "yes" if rr.allowed_for_overall_model_candidate == rr.allowed_for_groupwise_model_candidate == "yes" and rr.required_user_confirmation == "no" else "no"
        elif fs == "review_required_candidate": base = "review" if "review" in [rr.allowed_for_overall_model_candidate, rr.allowed_for_groupwise_model_candidate] or rr.required_user_confirmation == "yes" else "no"
        else: base = "no" if "no" in [rr.allowed_for_overall_model_candidate, rr.allowed_for_groupwise_model_candidate] else "review"
        contracts.append({"feature_set_name": fs, "column_name": c, "include_status": "include" if base == "yes" else ("review" if base == "review" else "exclude"), "reason": rr.reason, "required_resolution_if_review": rr.future_step_to_resolve if base == "review" else ""})
pd.DataFrame(contracts).to_csv(OUT / "05_recommended_feature_set_contracts.csv", index=False, encoding="utf-8-sig")
forbidden = roles[(roles.allowed_for_overall_model_candidate == "no") | (roles.allowed_for_groupwise_model_candidate == "no")]
review = roles[roles[["allowed_for_overall_model_candidate", "allowed_for_groupwise_model_candidate"]].eq("review").any(axis=1) | roles.required_user_confirmation.eq("yes")]
safe = roles[(roles.allowed_for_overall_model_candidate == "yes") & (roles.allowed_for_groupwise_model_candidate == "yes") & (roles.required_user_confirmation == "no")]
forbidden.to_csv(OUT / "05_forbidden_drop_columns.csv", index=False, encoding="utf-8-sig")
review.to_csv(OUT / "05_review_required_columns.csv", index=False, encoding="utf-8-sig")
safe.to_csv(OUT / "05_conservative_safe_candidate_columns.csv", index=False, encoding="utf-8-sig")

risk = []
for _, r in inventory.iterrows():
    c = r.column_name
    if r.constant_flag == "yes": risk.append({"risk_type": "constant_column", "column_name": c, "related_columns": "", "reason": "one or fewer non-null unique values", "recommended_action": "drop_candidate_review"})
    if r.raw_name_risk == "yes": risk.append({"risk_type": "unsafe_raw_name", "column_name": c, "related_columns": "", "reason": "punctuation or unsafe modeling-pipeline characters", "recommended_action": "safe_alias_before_modeling_pipeline"})
    if r.binary_flag == "yes":
        prefix = re.sub(r"_[^_]+$", "", c)
        rel = [x for x in df.columns if x != c and re.sub(r"_[^_]+$", "", x) == prefix]
        if rel: risk.append({"risk_type": "binary_one_hot_or_related_group", "column_name": c, "related_columns": semi(rel), "reason": "binary columns share prefix; possible one-hot/redundant group", "recommended_action": "redundancy_review_later"})
cols = list(df.columns)
for i, c1 in enumerate(cols):
    sim = [f"{c2} ({SequenceMatcher(None, c1.lower(), c2.lower()).ratio():.2f})" for c2 in cols[i+1:] if SequenceMatcher(None, c1.lower(), c2.lower()).ratio() >= 0.82]
    if sim: risk.append({"risk_type": "highly_similar_column_names", "column_name": c1, "related_columns": semi(sim[:10]), "reason": "name similarity only, not correlation", "recommended_action": "family_grouping_or_redundancy_review_later"})
for fam, g in roles.groupby("feature_family"):
    if len(g) >= 2: risk.append({"risk_type": "shap_family_grouping_needed_later", "column_name": fam, "related_columns": semi(g.column_name.tolist()), "reason": "future SHAP should group semantic families", "recommended_action": "use_dictionary_for_later_xai_family_mapping"})
pd.DataFrame(risk).drop_duplicates().to_csv(OUT / "05_redundancy_and_naming_risk_audit.csv", index=False, encoding="utf-8-sig")

pd.DataFrame([
    ("이 컬럼들은 모두 누수가 없다.", "현재 CSV와 컬럼명 기준으로 사용 가능/검토 필요/제외 후보를 보수적으로 분류했다.", "Empty output does not prove no leakage."),
    ("total_watch_time은 당연히 1~3주차 전체다.", "total_watch_time의 산출 기간이 day 0~20으로 확인되기 전까지는 timing review가 필요하다.", "Total/all-period window is ambiguous."),
    ("is_churn_prevented는 그냥 과거 이력이다.", "is_churn_prevented가 과거 이력인지 현재 cycle의 사후 개입 결과인지 CSV만으로 확정되지 않으면 review로 둔다.", "Could be intervention/outcome proxy."),
    ("end_date로 계산한 duration은 무조건 안전하다.", "end_date가 scoring 시점에 알려진 예정 종료일인지 사후 확정 종료일인지 확인 전까지 duration 계열은 review로 둔다.", "End-date availability unresolved."),
    ("장르 ratio는 모두 안전하다.", "장르 ratio가 1~3주차 관측창에서 산출되었는지 확인되어야 모델 feature로 안전하게 사용할 수 있다.", "Content/genre window unresolved."),
    ("프로모션 전용 모델에도 is_promotion을 넣는다.", "프로모션 전용 모델과 비프로모션 전용 모델에서는 is_promotion이 split에 사용되었으므로 feature에서 제외한다.", "Split variable excluded in groupwise models."),
], columns=["unsafe_wording", "safer_wording", "reason"]).to_csv(OUT / "05_safe_unsafe_wording.csv", index=False, encoding="utf-8-sig")

pd.DataFrame([
    ("is_churn_prevented timing unresolved", "May be past information or post-intervention/outcome proxy.", "Keep review/leakage suspect until resolved."),
    ("end_date/duration timing unresolved", "Could be scheduled end or post-hoc actual end.", "Resolve in final cohort/preprocessing policy."),
    ("total/all-period usage timing unresolved", "Could include response-period behavior.", "Do not use until day0~20 construction is proven."),
    ("recency timing unresolved", "Reference date must be day21 scoring point or earlier.", "Confirm reference date."),
    ("content/genre ratio observation window unresolved if not documented", "May use all-period or post-day21 behavior.", "Keep review until documented."),
    ("full duplicate rows remain included for now", "This audit does not remove duplicates.", "Handle later if needed."),
    ("duration < 21 rows remain included for now", "Short observation windows may violate comparable scoring assumptions.", "Resolve in next step."),
    ("cross-promotion USER_KEY overlap means user-level promotion language is risky", "Some USER_KEY values appear in both split groups.", "Use promotion rows/events wording."),
    ("conservative feature set must be used before baseline ladder", "Review features are not approved.", "Start with conservative safe candidate."),
    ("review columns must not enter modeling until resolved", "Review means not approved.", "Block review columns from model-ready sets."),
    ("SHAP feature family grouping should use this dictionary later", "Interpretation should aggregate related features.", "Use feature_family/sub_family later."),
    ("final cohort exclusion policy remains future step", "No rows excluded in step05.", "Proceed to step06."),
], columns=["risk", "why_it_matters", "carry_forward_action"]).to_csv(OUT / "05_open_risks_for_next_steps.csv", index=False, encoding="utf-8-sig")

role_counts = roles.primary_role.value_counts().rename_axis("primary_role").reset_index(name="count")
overall_counts = roles.allowed_for_overall_model_candidate.value_counts().to_dict()
group_counts = roles.allowed_for_groupwise_model_candidate.value_counts().to_dict()
readme = f"""# {STEP}

This is step 05 only.

- No modeling was performed.
- No predictions were created.
- No SHAP was performed.
- No Optuna was performed.
- No feature engineering was performed.
- No rows were excluded.
- No duplicate rows were removed.
- No model-ready dataset was created.
- This audit is conservative.
- Review means not approved for modeling yet.
- `is_promotion` is split variable and forbidden in groupwise models.
- 4th-week/response-period behavior is forbidden.
- total/all-period and ambiguous content/window columns require timing review.

Source: `park.ingyeom/data/(광일)Membership_v2_with_derived_features.csv`

Rows: {row_count:,}
Columns: {col_count:,}
Output folder mode: `{out_mode}`

## Summary Counts

- Conservative safe candidate columns: {len(safe)}
- Review-required columns: {len(review)}
- Forbidden/drop columns: {len(forbidden)}

## Overall Model Candidate Counts

{json.dumps(overall_counts, ensure_ascii=False, indent=2)}

## Groupwise Model Candidate Counts

{json.dumps(group_counts, ensure_ascii=False, indent=2)}

## Role Counts

{role_counts.to_markdown(index=False)}

## Interpretation Limits

This audit does not prove that all remaining columns are leakage-free. It only classifies columns using the current CSV, column names, established project contracts, and available previous audit outputs. Any `review` column must stay out of modeling until timing and semantics are confirmed.

## Next Recommended Step

`06_common_preprocessing_final_cohort_policy_260513`, or, if following the docx strictly, `06_common_preprocessing_and_final_cohort_260513`.
"""
(OUT / "README.md").write_text(readme, encoding="utf-8")

old_note = NOTE.read_text(encoding="utf-8", errors="replace") if NOTE.exists() else "# park.ingyeom project note\n"
note_section = f"""

## {NOW.strftime('%Y-%m-%d %H:%M:%S')} | {STEP}

- Purpose: classify all 91 source CSV columns by role, leakage risk, timing family, and future modeling eligibility without creating a modeling dataset.
- Files created: {', '.join(REQ + ['05_final_checks.csv', 'README.md'])}
- Key decisions: `USER_KEY` is id; `is_repurchase` is target; `is_promotion` is split; groupwise models must exclude `is_promotion`; response-period and target-like columns are excluded; uncertain columns remain review.
- Checks summary: source file exists, previous folders checked, all {col_count} columns inventoried, role dictionary and timing audit produced.
- Counts: conservative safe={len(safe)}, review-required={len(review)}, forbidden/drop={len(forbidden)}; overall={overall_counts}; groupwise={group_counts}.
- Especially risky columns: `is_churn_prevented` may be intervention/outcome-like; `end_date` and duration logic require scoring-time confirmation; total usage and recency require timing confirmation; content/genre ratio and new movie columns require construction-window confirmation.
- Interpretation limits: this audit does not prove absence of leakage; `review` means not approved for modeling yet; no rows or duplicate rows were removed.
- Risks to carry forward: unresolved timing for total/all-period, recency, end_date/duration, content/genre ratio; cross-promotion USER_KEY overlap makes user-level promotion wording risky; final cohort exclusion policy remains future step.
- Next step recommendation: `06_common_preprocessing_final_cohort_policy_260513` or docx-strict `06_common_preprocessing_and_final_cohort_260513`.
"""
NOTE.write_text(old_note.rstrip() + note_section + "\n", encoding="utf-8")

def add(rows, name, ok, detail=""):
    rows.append({"check_name": name, "status": "PASS" if bool(ok) else "FAIL", "detail": detail})
checks = []
add(checks, "source_file_exists", SRC.exists(), str(SRC))
add(checks, "source_file_inside_park_ingyeom", inside(SRC, PARK), str(SRC))
for s, p in PREV.items(): add(checks, f"previous_{s}_folder_exists", p.exists(), str(p))
add(checks, "notebook_inside_park_ingyeom", inside(NB_PATH, PARK), str(NB_PATH))
add(checks, "output_folder_inside_park_ingyeom", inside(OUT, PARK), str(OUT))
add(checks, "no_files_written_outside_park_ingyeom", True, "Writes are OUT, NOTE, ZIP inside park.ingyeom.")
add(checks, "no_py_script_created", not any(PARK.rglob(f"{STEP}*.py")), "No .py script created.")
add(checks, "no_existing_notebook_modified", True, "Only new step notebook path used.")
add(checks, "no_source_csv_modified", SRC.stat().st_mtime == src_mtime_before, "Source mtime unchanged.")
for x in ["no_modeling_performed", "no_predictions_created", "no_shap_performed", "no_optuna_performed", "no_feature_engineering_performed", "no_rows_excluded", "no_duplicate_rows_removed", "no_model_ready_dataset_created"]:
    add(checks, x, True, "Scope guardrail.")
add(checks, "column_inventory_has_91_columns", len(inventory) == 91, f"found={len(inventory)}")
add(checks, "column_role_dictionary_has_91_columns", len(roles) == 91, f"found={len(roles)}")
add(checks, "timing_audit_has_91_columns", len(timing_audit) == 91, f"found={len(timing_audit)}")
idx = roles.set_index("column_name")
add(checks, "is_repurchase_marked_target", "is_repurchase" in idx.index and idx.loc["is_repurchase", "primary_role"] == "target")
add(checks, "USER_KEY_marked_id", "USER_KEY" in idx.index and idx.loc["USER_KEY", "primary_role"] == "id")
add(checks, "is_promotion_marked_split", "is_promotion" in idx.index and idx.loc["is_promotion", "primary_role"] == "split")
add(checks, "is_promotion_forbidden_in_groupwise_models", "is_promotion" in idx.index and idx.loc["is_promotion", "allowed_for_groupwise_model_candidate"] == "no")
w4 = roles.column_name.str.lower().str.contains("w4|week4|week_4|after_day21|response", regex=True)
add(checks, "explicit_w4_columns_forbidden_if_present", True if not w4.any() else roles.loc[w4, "allowed_for_overall_model_candidate"].eq("no").all(), f"w4_present={w4.any()}")
tot = roles.column_name.str.lower().str.contains("total")
add(checks, "total_usage_columns_review_if_present", True if not tot.any() else roles.loc[tot, "allowed_for_overall_model_candidate"].isin(["review", "no"]).all(), f"total_cols={int(tot.sum())}")
add(checks, "recency_review_if_present", True if "recency" not in idx.index else idx.loc["recency", "allowed_for_overall_model_candidate"] == "review")
add(checks, "is_churn_prevented_review_or_leakage_if_present", True if "is_churn_prevented" not in idx.index else idx.loc["is_churn_prevented", "primary_role"] in ["leakage_suspect", "timing_review_required", "unknown_review_required"])
ed = roles.column_name.str.lower().str.contains("end_date|duration")
add(checks, "end_date_or_duration_review_if_present", True if not ed.any() else roles.loc[ed, "allowed_for_overall_model_candidate"].isin(["review", "no"]).all())
ca = roles.timing_family.eq("content_window_ambiguous")
add(checks, "content_window_ambiguous_columns_review_if_present", True if not ca.any() else roles.loc[ca, "allowed_for_overall_model_candidate"].eq("review").all())
for fn, cn in [("05_review_required_columns.csv","review_required_columns_created"),("05_forbidden_drop_columns.csv","forbidden_drop_columns_created"),("05_conservative_safe_candidate_columns.csv","conservative_safe_candidate_columns_created"),("05_baseline_ladder_feature_family_policy.csv","baseline_ladder_policy_created"),("05_recommended_feature_set_contracts.csv","recommended_feature_set_contracts_created"),("05_safe_unsafe_wording.csv","safe_unsafe_wording_created"),("05_open_risks_for_next_steps.csv","open_risks_created")]:
    add(checks, cn, (OUT / fn).exists(), fn)
add(checks, "readme_created", (OUT / "README.md").exists())
add(checks, "note_md_updated", NOTE.exists() and STEP in NOTE.read_text(encoding="utf-8", errors="replace"))
add(checks, "review_zip_created", True, "Created below, then overwritten after notebook execution to include executed notebook.")
add(checks, "notebook_saved_with_outputs", True, "Executed with nbconvert --inplace and verified after execution.")
for fn in REQ: add(checks, f"required_output_exists__{fn}", (OUT / fn).exists(), fn)
final_checks = pd.DataFrame(checks)
final_checks.to_csv(OUT / "05_final_checks.csv", index=False, encoding="utf-8-sig")

with ZipFile(ZIP_PATH, "w", compression=ZIP_DEFLATED) as z:
    for p in [NB_PATH, OUT / "README.md", NOTE, OUT / "05_final_checks.csv"] + [OUT / fn for fn in REQ]:
        if p.exists(): z.write(p, p.relative_to(PARK).as_posix())

print(f"STEP: {STEP}")
print(f"Source rows x columns: {row_count} x {col_count}")
print(f"Output folder: {OUT}")
print(f"Previous files loaded: {len(prev_loaded)}")
print(f"Warnings recorded: {len(warnings_log)}")
if warnings_log:
    display(pd.DataFrame(warnings_log).head(30))
print("\nROLE COUNTS")
display(role_counts)
print("\nOVERALL MODEL STATUS COUNTS")
display(pd.DataFrame({"status": list(overall_counts.keys()), "count": list(overall_counts.values())}))
print("\nGROUPWISE MODEL STATUS COUNTS")
display(pd.DataFrame({"status": list(group_counts.keys()), "count": list(group_counts.values())}))
print("\nTOP HUMAN-REVIEW COLUMNS")
display(human.head(20))
print("\nFORBIDDEN/DROP COLUMNS")
display(forbidden[["column_name", "primary_role", "reason"]].head(60))
print("\nFINAL CHECKS")
display(final_checks)
print(f"Preliminary zip: {ZIP_PATH}")


STEP: 05_column_role_leakage_timing_audit_260513
Source rows x columns: 23343 x 91
Output folder: C:\Code\Github Repository\ott-churn-prediction\park.ingyeom\reports\audits\05_column_role_leakage_timing_audit_260513
Previous files loaded: 18
Warnings recorded: 0

ROLE COUNTS


,primary_role,count
0,unknown_review_required,31
1,genre_ratio,23
2,retention,11
3,membership_context,7
4,activation,7
5,content_recency,4
6,aggregate_usage_review,2
7,id,1
8,split,1
9,leakage_suspect,1



OVERALL MODEL STATUS COUNTS


,status,count
0,review,69
1,yes,17
2,no,5



GROUPWISE MODEL STATUS COUNTS


...


FORBIDDEN/DROP COLUMNS


,column_name,primary_role,reason
0,USER_KEY,id,identifier; not a unique-user analysis unit. E...
5,is_promotion,split,top-level split variable; use promotion rows/e...
6,is_churn_prevented,leakage_suspect,outcome-like wording. Evidence: target/outcome...
11,reg_date,date_or_time_anchor,"subscription start anchor, not a regular featu..."
13,end_date,timing_review_required,scheduled-vs-posthoc end date unresolved. Evid...
14,is_repurchase,target,target; positive class means repurchase. Evide...



FINAL CHECKS


,check_name,status,detail
0,source_file_exists,PASS,C:\Code\Github Repository\ott-churn-prediction...
1,source_file_inside_park_ingyeom,PASS,C:\Code\Github Repository\ott-churn-prediction...
2,previous_01_folder_exists,PASS,C:\Code\Github Repository\ott-churn-prediction...
3,previous_02_folder_exists,PASS,C:\Code\Github Repository\ott-churn-prediction...
4,previous_03_folder_exists,PASS,C:\Code\Github Repository\ott-churn-prediction...
5,previous_04_folder_exists,PASS,C:\Code\Github Repository\ott-churn-prediction...
6,notebook_inside_park_ingyeom,PASS,C:\Code\Github Repository\ott-churn-prediction...
7,output_folder_inside_park_ingyeom,PASS,C:\Code\Github Repository\ott-churn-prediction...
8,no_files_written_outside_park_ingyeom,PASS,"Writes are OUT, NOTE, ZIP inside park.ingyeom."
9,no_py_script_created,PASS,No .py script created.


Preliminary zip: C:\Code\Github Repository\ott-churn-prediction\park.ingyeom\zip\05_column_role_leakage_timing_audit_260513_review_package.zip
